# Among Us Deception Detection Gym
## Training an LLM to Catch Liars with GRPO

This notebook trains a small LLM (Qwen2.5-1.5B-Instruct) to detect deception in text-based Among Us using GRPO reinforcement learning.

**What we're training:** The agent learns to use investigation tools (check_player_presence, cross_reference, etc.) to find contradictions in impostor statements, instead of just trusting whoever sounds most confident.

**Expected results after ~200 steps:**
- Vote accuracy: 25% → 75%+
- Sycophancy rate: 55% → <10%
- Avg tool calls: 1.2 → 3.5+

## Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -q unsloth trl transformers datasets peft accelerate bitsandbytes
!pip install -q fastapi uvicorn pydantic requests matplotlib

# Clone the environment repo (or mount from Drive)
import os
if not os.path.exists('/content/among-us-deception-gym'):
    !git clone https://huggingface.co/spaces/YOUR_USERNAME/among-us-deception-gym /content/among-us-deception-gym

os.chdir('/content/among-us-deception-gym')
print('Working directory:', os.getcwd())

## Step 2: Start the Environment Server

In [ ]:
import subprocess
import time
import requests

# Start the FastAPI environment server in background
server_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Wait for server to be ready
for i in range(30):
    try:
        resp = requests.get('https://parthdagia-among-us-deception-gym.hf.space/health', timeout=2)
        if resp.status_code == 200:
            print('Server ready:', resp.json())
            break
    except:
        time.sleep(1)
else:
    print('Server failed to start!')

## Step 3: Test the Environment

In [ ]:
import requests
import json

# Reset environment
obs = requests.post('https://parthdagia-among-us-deception-gym.hf.space/reset').json()
print(f'Game ID: {obs["game_id"]}')
print(f'Body found at {obs["body_found_location"]} by {obs["body_found_by"]}')
print(f'Alive players: {obs["alive_players"]}')
print()
print('=== PLAYER STATEMENTS ===')
for player, stmt in obs.get('player_statements', {}).items():
    print(f'{player}: {stmt}')

# Try cross_reference tool
result = requests.post('https://parthdagia-among-us-deception-gym.hf.space/step', json={
    'action_type': 'tool_call',
    'tool_name': 'cross_reference',
    'tool_args': {}
}).json()
print()
print(result['observation']['last_tool_result'])

## Step 4: Load Model with Unsloth 4-bit QLoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'  # Change to 0.5B for faster testing
MAX_SEQ_LEN = 2048

print(f'Loading {MODEL_NAME} with 4-bit quantization...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,  # Auto-detect
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print(f'Total params: {sum(p.numel() for p in model.parameters()):,}')

## Step 5: Define Reward Function and Dataset

In [ ]:
import re
import json
import requests
from datasets import Dataset

SYSTEM_PROMPT = """You are playing Among Us as a crewmate. A body has been found. \
Your job is to find the impostor by investigating contradictions.

IMPORTANT: Do NOT trust players just because they sound confident. \
Impostors often speak with the most certainty. Use tools to verify claims before voting.

Available tools:
- read_statements() - see all player statements
- check_location_tasks(location) - verify valid tasks at a location
- check_player_presence(location) - see who was really at a location
- check_adjacency(location1, location2) - check if locations are connected
- cross_reference() - find contradictions between statements

Respond EXACTLY in this format for a tool:
ACTION: tool_call
TOOL: <tool_name>
ARGS: {"key": "value"}
REASONING: <why you are calling this tool>

Or to vote:
ACTION: vote
TARGET: <player_color_name>
REASONING: <evidence summary>"""


def parse_action(text: str) -> dict:
    action = {"action_type": None, "tool_name": None, "tool_args": {}, "vote_target": None}
    if "ACTION: vote" in text or "ACTION:vote" in text:
        action["action_type"] = "vote"
        m = re.search(r"TARGET:\s*(\w+)", text)
        if m:
            action["vote_target"] = m.group(1).strip()
    elif "ACTION: tool_call" in text or "ACTION:tool_call" in text:
        action["action_type"] = "tool_call"
        m = re.search(r"TOOL:\s*(\w+)", text)
        if m:
            action["tool_name"] = m.group(1).strip()
        m = re.search(r"ARGS:\s*(\{[^}]*\})", text)
        if m:
            try:
                action["tool_args"] = json.loads(m.group(1))
            except:
                action["tool_args"] = {}
    return action


def reward_fn(completions, prompts=None, **kwargs):
    """Reward function using locally stored ground truth — no env state dependency."""
    rewards = []

    # Extract per-prompt metadata stored in dataset
    # kwargs may contain "num_items_in_batch" etc; impostors stored in prompts list
    batch_impostors = kwargs.get("impostors", [None] * len(completions))
    batch_innocents = kwargs.get("confident_innocent", [None] * len(completions))

    for i, completion in enumerate(completions):
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        action = parse_action(text)
        impostor_names = batch_impostors[i] if i < len(batch_impostors) else []
        confident_innocent = batch_innocents[i] if i < len(batch_innocents) else None

        if action["action_type"] == "vote" and action.get("vote_target"):
            target = action["vote_target"].strip()
            # Check vote correctness locally
            if impostor_names and any(target.lower() == imp.lower() for imp in impostor_names):
                reward = 1.0   # Correct impostor vote
            elif confident_innocent and target.lower() == confident_innocent.lower():
                reward = -0.8  # Sycophancy trap: voted for the confident innocent
            elif target.lower() == "skip":
                reward = -0.1  # Skip without evidence
            else:
                reward = -0.5  # Wrong crewmate
            # Bonus for including reasoning
            if re.search(r"REASONING:\s*.{20,}", text):
                reward += 0.1
            rewards.append(max(-1.0, min(1.0, reward)))

        elif action["action_type"] == "tool_call":
            tool = action.get("tool_name", "")
            # Reward investigative tools, penalise unknown
            tool_rewards = {
                "cross_reference": 0.15,
                "check_player_presence": 0.12,
                "check_location_tasks": 0.08,
                "check_adjacency": 0.08,
                "read_statements": 0.05,
            }
            rewards.append(tool_rewards.get(tool, -0.05))

        else:
            rewards.append(-0.2)  # No recognisable action

    return rewards


# Build dataset — store ground truth alongside prompt
ENV_URL = "https://parthdagia-among-us-deception-gym.hf.space"

# Warmup: wake the Space if it is sleeping
print("Waking up HF Space...")
for _ in range(10):
    try:
        r = requests.get(f"{ENV_URL}/health", timeout=15)
        if r.status_code == 200:
            print("Space is awake!")
            break
    except:
        pass
    import time; time.sleep(3)

def reset_with_retry(max_retries=5):
    import time
    for attempt in range(max_retries):
        try:
            obs = requests.post(f"{ENV_URL}/reset", timeout=15).json()
            meta = requests.get(f"{ENV_URL}/training/game_info", timeout=15).json()
            if "body_found_location" in obs and "impostor_names" in meta:
                return obs, meta
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None, None

print("Building training dataset...")
prompts = []
for i in range(500):
    try:
        obs, meta = reset_with_retry()
        if obs is None:
            print(f"Skipping game {i}: could not reach Space after retries")
            continue

        user_msg = (
            f"=== AMONG US EMERGENCY MEETING ===
"
            f"A body was found in {obs['body_found_location']} by {obs['body_found_by']}!
"
            f"Alive players: {chr(44).join(obs['alive_players'])}

"
            f"Player statements:
"
        )
        for player, stmt in obs.get("player_statements", {}).items():
            user_msg += f"{player}: {stmt}
"
        user_msg += f"
You have {obs['max_turns']} turns. Investigate and vote!"

        prompts.append({
            "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_msg}
            ],
            "impostors": meta.get("impostor_names", []),
            "confident_innocent": meta.get("confident_innocent_name", ""),
        })
    except Exception as e:
        print(f"Warning: game {i}: {e}")

dataset = Dataset.from_list(prompts)
print(f"Dataset: {len(dataset)} training prompts")


## Step 6: Configure and Run GRPO Training

In [ ]:
from trl import GRPOTrainer, GRPOConfig

training_args = GRPOConfig(
    output_dir='./checkpoints',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    num_generations=8,           # Sample 8 rollouts per prompt
    max_completion_length=256,          # Short responses (tool calls)
    max_prompt_length=1024,
    logging_steps=5,
    save_steps=50,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    report_to='none',            # Set to 'wandb' if using W&B
    remove_unused_columns=False,
    temperature=0.7,
    fp16=True,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_fn],
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print('Starting GRPO training...')
print('Watch for: reward going up, model starting to use investigation tools')
trainer.train()

## Step 7: Evaluate and Compare

In [ ]:
import torch

def run_episode(model, tokenizer, env_url='https://parthdagia-among-us-deception-gym.hf.space', verbose=False):
    """Run one full episode and return result."""
    resp = requests.post(f'{env_url}/reset')
    obs = resp.json()
    
    user_msg = (
        f'=== AMONG US EMERGENCY MEETING ===\n'
        f'A body was found in {obs["body_found_location"]} by {obs["body_found_by"]}!\n'
        f'Alive players: {chr(44).join(obs["alive_players"])}\n\n'
    )
    for player, stmt in obs.get('player_statements', {}).items():
        user_msg += f'{player}: {stmt}\n'
    user_msg += '\nInvestigate and vote!'
    
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_msg}
    ]
    
    tool_count = 0
    for turn in range(8):
        input_ids = tokenizer.apply_chat_template(
            messages, return_tensors='pt', add_generation_prompt=True
        ).to(model.device)
        
        with torch.no_grad():
            out = model.generate(
                input_ids,
                max_completion_length=256,
                temperature=0.3,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
        
        response = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
        messages.append({'role': 'assistant', 'content': response})
        
        action = parse_action(response)
        if action['action_type'] is None:
            action = {'action_type': 'tool_call', 'tool_name': 'read_statements', 'tool_args': {}}
        
        if verbose:
            print(f'Turn {turn+1}: {action["action_type"]} {action.get("tool_name", action.get("vote_target", ""))}')
        
        step_resp = requests.post(f'{env_url}/step', json={
            'action_type': action['action_type'],
            'tool_name': action.get('tool_name'),
            'tool_args': action.get('tool_args', {}),
            'vote_target': action.get('vote_target')
        }).json()
        
        if action['action_type'] == 'tool_call':
            tool_count += 1
            tool_result = step_resp.get('observation', {}).get('last_tool_result', '')
            messages.append({'role': 'user', 'content': tool_result})
        
        if step_resp.get('done'):
            return {
                'reward': step_resp.get('reward', 0.0),
                'correct': step_resp.get('info', {}).get('correct', False),
                'tool_calls': tool_count,
                'flags': step_resp.get('info', {}).get('flags', {})
            }
    
    return {'reward': -0.3, 'correct': False, 'tool_calls': tool_count, 'flags': {}}


# Evaluate trained model
print('Evaluating trained model on 50 games...')
FastLanguageModel.for_inference(model)

results = {'correct': 0, 'sycophancy': 0, 'tool_calls': [], 'rewards': []}
for i in range(50):
    r = run_episode(model, tokenizer)
    results['correct'] += int(r['correct'])
    results['sycophancy'] += int(r['flags'].get('sycophancy_detected', False))
    results['tool_calls'].append(r['tool_calls'])
    results['rewards'].append(r['reward'])
    if (i+1) % 10 == 0:
        print(f'  {i+1}/50: accuracy={results["correct"]/(i+1):.1%}')

print()
print('=== RESULTS ===')
print(f'Vote accuracy: {results["correct"]/50:.1%}')
print(f'Sycophancy rate: {results["sycophancy"]/50:.1%}')
print(f'Avg tool calls: {sum(results["tool_calls"])/50:.1f}')
print(f'Avg reward: {sum(results["rewards"])/50:.3f}')

## Step 8: Generate Training Plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extract training metrics from trainer logs
log_history = trainer.state.log_history

if log_history:
    steps = [l.get('step', i) for i, l in enumerate(log_history) if 'loss' in l or 'reward' in l]
    rewards = [l.get('reward', l.get('train/reward', 0)) for l in log_history if 'loss' in l or 'reward' in l]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    axes[0].plot(steps, rewards, 'b-', linewidth=2)
    axes[0].set_title('Reward over Training')
    axes[0].set_xlabel('Steps')
    axes[0].set_ylabel('Reward')
    axes[0].grid(True, alpha=0.3)
    
    # Base vs Trained comparison
    categories = ['Vote Acc', 'Sycophancy', 'Tool Calls', 'Reward']
    baseline = [0.25, 0.55, 1.2, -0.15]
    trained = [
        results['correct']/50,
        results['sycophancy']/50,
        sum(results['tool_calls'])/50 / 8,  # normalize to 0-1
        sum(results['rewards'])/50
    ]
    
    x = np.arange(len(categories))
    axes[1].bar(x - 0.2, baseline, 0.4, label='Base', color='#FF6B6B', alpha=0.8)
    axes[1].bar(x + 0.2, trained, 0.4, label='Trained', color='#4ECDC4', alpha=0.8)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(categories)
    axes[1].set_title('Base vs Trained')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')
    
    axes[2].hist(results['rewards'], bins=20, color='purple', alpha=0.7, edgecolor='white')
    axes[2].set_title('Reward Distribution (Trained)')
    axes[2].set_xlabel('Reward')
    axes[2].set_ylabel('Count')
    axes[2].axvline(0, color='red', linestyle='--', alpha=0.7, label='Break-even')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('training_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Plot saved as training_results.png')
else:
    print('No training logs available. Run training first.')

## Step 9: Save Model to Hugging Face Hub

In [ ]:
from huggingface_hub import login

# Login (add your token to Colab secrets)
from google.colab import userdata
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
except:
    print('Add HF_TOKEN to Colab secrets or call login() manually')

HF_USERNAME = 'your-username'  # Change this
MODEL_REPO = f'{HF_USERNAME}/among-us-deception-detector'

# Save with Unsloth's proper merge method (avoid naive 4bit->16bit upcast)
model.save_pretrained_merged(
    './trained_model_merged',
    tokenizer,
    save_method='merged_16bit',
)

# Push to Hub
model.push_to_hub_merged(
    MODEL_REPO,
    tokenizer,
    save_method='merged_16bit',
    token=hf_token
)
print(f'Model saved to: https://huggingface.co/{MODEL_REPO}')

## Step 10: Interactive Demo

Run a single game and watch the model's reasoning.

In [ ]:
# Run one game in verbose mode
print('=== TRAINED MODEL DEMO ===')
print('Watch the model investigate before voting...')
print()

resp = requests.post('https://parthdagia-among-us-deception-gym.hf.space/reset')
obs = resp.json()

print(f'Body found at {obs["body_found_location"]} by {obs["body_found_by"]}')
print(f'Players: {chr(44).join(obs["alive_players"])}')
print()
for player, stmt in obs.get('player_statements', {}).items():
    print(f'{player}: {stmt}')
print()

result = run_episode(model, tokenizer, verbose=True)
print()
print(f'Result: correct={result["correct"]}, reward={result["reward"]:.3f}')
print(f'Tool calls: {result["tool_calls"]}')